[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/01_annotation_label_noise/01_annotation_label_noise.ipynb)

# 01 · 标注与标签噪声（从零实现）

目标：把**标签噪声**变成一个可建模、可估计、可校正的对象。用 numpy 从零：① 生成对称/类条件噪声；② 推导并验证噪声给 accuracy 设的**天花板**；③ 看噪声如何**扭曲模型排名**；④ 从标注者分歧**估计噪声率**；⑤ 用**置信学习**思路估转移矩阵并定位错标；⑥ 用 T 做**统计校正**。

路线：噪声生成 → accuracy 天花板 → 排名扭曲 → 噪声率估计 → 置信学习去噪 → 校正 → ✏️ 练习 → 📖 答案 → 🧪 真实仇恨言论标注胶囊。

> 纪律：所有随机用 `default_rng(seed)` 固定；所有「应成立的性质」写成 `assert`（生成的 T 行和为 1、完美模型测得 acc = 1−ε、估出的 ε 接近真值……）。

## 0 · 数据 helper（联网取真实数据，失败回退）

下面这个 cell 定义全课统一的下载工具，最后的真实数据胶囊会用到。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    '''下载 url；给 fn 则缓存到本地文件并返回路径，否则返回 bytes。'''
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    '''HuggingFace datasets-server 分页取真实数据行，返回 list[dict]。'''
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · 对称噪声：最简单的脏法

**对称噪声**：每个真标签以概率 `eps` 被均匀翻成其它任一类。转移矩阵 `T[i,j]`：对角线 `1-eps`，非对角各 `eps/(K-1)`。

我们先造转移矩阵，再用它给一批干净标签**注入噪声**。

In [ ]:
def symmetric_T(K, eps):
    '''K 类对称噪声的转移矩阵：对角 1-eps，非对角 eps/(K-1)。'''
    T = np.full((K, K), eps / (K - 1))
    np.fill_diagonal(T, 1 - eps)
    return T

def inject_noise(y, T, seed=0):
    '''按转移矩阵 T 给真标签 y 注入噪声，返回观测标签 y_tilde。'''
    r = np.random.default_rng(seed)
    y = np.asarray(y)
    y_tilde = np.empty_like(y)
    for i in range(len(y)):
        y_tilde[i] = r.choice(T.shape[1], p=T[y[i]])   # 从真类 y[i] 那一行抽观测类
    return y_tilde

K, eps = 3, 0.2
T = symmetric_T(K, eps)
print('对称转移矩阵 T (eps=0.2, K=3):'); print(T)
assert np.allclose(T.sum(axis=1), 1.0), '每行(给定真类的观测分布)必须和为 1'
assert np.allclose(np.diag(T), 1 - eps)

rng = np.random.default_rng(0)
y = rng.integers(0, K, size=20000)
y_tilde = inject_noise(y, T, seed=1)
emp_flip = (y != y_tilde).mean()
print(f'\n注入后实测翻转率 = {emp_flip:.3f}  (理论 eps = {eps})')
assert abs(emp_flip - eps) < 0.01, '实测翻转率应接近 eps'
print('✅ 对称噪声生成正确：T 行随机、实测翻转率 ≈ eps')

## 2 · 类条件噪声：更真实的脏法

现实里错误**有方向**：某些类更容易被标错、错向某个特定类。这由一个**任意行随机矩阵** T 刻画（每行可不同）。

造一个「中性常被误标为负面」的情感三分类噪声矩阵，并验证观测分布 = 真分布 · T。

In [ ]:
# 类: 0=负面 1=中性 2=正面。中性(1)有 30% 概率被标成负面(0)，是主要错误方向
T_cc = np.array([
    [0.90, 0.07, 0.03],   # 真负面: 大多标对
    [0.30, 0.62, 0.08],   # 真中性: 30% 被误标为负面!
    [0.05, 0.10, 0.85],   # 真正面: 大多标对
])
assert np.allclose(T_cc.sum(axis=1), 1.0)

# 真实类别分布 pi（中性最多）
pi = np.array([0.3, 0.5, 0.2])
N = 50000
y = rng.choice(3, size=N, p=pi)
y_tilde = inject_noise(y, T_cc, seed=2)

# 观测标签分布应 ≈ pi @ T_cc
obs_dist = np.bincount(y_tilde, minlength=3) / N
pred_dist = pi @ T_cc
print('真实类别分布 pi      :', pi)
print('观测标签分布(实测)   :', obs_dist)
print('预测 pi @ T_cc       :', pred_dist)
assert np.allclose(obs_dist, pred_dist, atol=0.01), '观测分布应等于 pi @ T'
print('\n✅ 类条件噪声: 观测分布 = pi @ T。注意负面类被噪声「灌水」变多了')

## 3 · 噪声给 accuracy 设的天花板

用含噪金标评测，一个**完美模型**(预测=真值)能测到多少 accuracy？只有当标签没被翻时它才「对」：

$$\text{测得 acc(完美模型)} = P(\tilde y = y) = \sum_i \pi_i T_{ii}$$

对称噪声下 = `1 - eps`。验证之。

In [ ]:
def perfect_model_acc(pi, T):
    '''完美模型(预测=真值)在含噪金标上的期望 accuracy = sum_i pi_i T_ii。'''
    return float(np.sum(pi * np.diag(T)))

# 对称噪声: 天花板应 = 1 - eps
for eps in [0.05, 0.1, 0.2, 0.3]:
    T = symmetric_T(3, eps)
    pi_uniform = np.ones(3) / 3
    ceil_theory = perfect_model_acc(pi_uniform, T)
    # 实测: 造数据, 完美模型预测=真值, 与含噪标签比
    y = rng.integers(0, 3, 30000); yt = inject_noise(y, T, seed=int(eps*100))
    ceil_emp = (y == yt).mean()      # 完美模型 pred=y
    print(f'eps={eps:.2f}: 理论天花板={ceil_theory:.3f}  实测={ceil_emp:.3f}  (=1-eps={1-eps:.2f})')
    assert abs(ceil_theory - (1 - eps)) < 1e-9
    assert abs(ceil_emp - ceil_theory) < 0.01
print('\n✅ 铁律: 含噪金标下，连完美模型也测不过 1-eps。噪声率 = accuracy 的天花板')

## 4 · 更阴险的：噪声扭曲模型排名

噪声不只整体压低分数，还会**扭曲模型间的相对排名**——某些模型的错误恰好与金标的错误「同向」，测得 accuracy 虚高。

模拟两个模型：A 真实能力更强但错误「随机」，B 真实能力略弱但其错误方向恰好与噪声同向。看含噪评测会不会把它们排反。

In [ ]:
def measured_acc(y_true, y_pred, T, seed):
    '''在含噪金标(由 y_true 经 T 生成)上测 y_pred 的 accuracy。'''
    yt = inject_noise(y_true, T, seed=seed)
    return (y_pred == yt).mean()

N = 40000
y_true = rng.integers(0, 3, N)
# 类条件噪声: 真中性(1) 常被标成负面(0)
T_cc = np.array([[0.95,0.03,0.02],[0.25,0.70,0.05],[0.03,0.05,0.92]])

# 模型 A: 真实 acc=0.80, 错误是随机翻到别的类
predA = y_true.copy(); mA = rng.random(N) < 0.20
predA[mA] = (predA[mA] + rng.integers(1,3,mA.sum())) % 3
# 模型 B: 真实 acc=0.78(更弱), 但它的错误偏向把中性也判成负面(与噪声同向)
predB = y_true.copy(); mB = rng.random(N) < 0.22
predB[mB & (y_true==1)] = 0      # 中性->负面, 蹭噪声
other = mB & (y_true!=1); predB[other] = (predB[other]+1)%3

true_accA = (predA==y_true).mean(); true_accB = (predB==y_true).mean()
meas_accA = measured_acc(y_true, predA, T_cc, 7)
meas_accB = measured_acc(y_true, predB, T_cc, 7)
print(f'真实能力 : A={true_accA:.3f}  B={true_accB:.3f}  -> A 更强')
print(f'含噪测得 : A={meas_accA:.3f}  B={meas_accB:.3f}  -> {"B 反超!" if meas_accB>meas_accA else "A 仍领先"}')
assert true_accA > true_accB, '构造上 A 真实更强'
print('\n观察: B 的错误蹭上了噪声方向，含噪评测可能把真实更弱的 B 排到前面 —— 噪声会让你选错模型')

## 5 · 从标注者分歧估计噪声率

没有干净子集时，**多标注者的分歧**本身携带噪声信息。对称二分类噪声率 `eps` 下，两个独立标注者对同一样本**不一致**的概率：

$$P(\text{disagree}) = 2\varepsilon(1-\varepsilon)$$

(一个翻一个没翻)。测出不一致率，解一元二次就得 `eps`。

In [ ]:
def estimate_eps_from_disagreement(labels_2raters):
    '''labels_2raters: shape (N, 2) 两个标注者对 N 个样本的二值标注。
       用 P(disagree)=2*eps*(1-eps) 反解 eps (取 <=0.5 的根)。'''
    disagree = (labels_2raters[:, 0] != labels_2raters[:, 1]).mean()
    # 2 eps^2 - 2 eps + disagree = 0  -> eps = (1 - sqrt(1-2*disagree))/2
    disc = max(0.0, 1 - 2 * disagree)
    eps_hat = (1 - np.sqrt(disc)) / 2
    return eps_hat, disagree

# 造数据: 真二值标签, 两个独立标注者各自以 eps 对称翻转
true_eps = 0.15
y = rng.integers(0, 2, 30000)
Tb = symmetric_T(2, true_eps)
r1 = inject_noise(y, Tb, seed=11)
r2 = inject_noise(y, Tb, seed=12)
labels2 = np.stack([r1, r2], axis=1)

eps_hat, dis = estimate_eps_from_disagreement(labels2)
print(f'实测不一致率 = {dis:.3f}  (理论 2*eps*(1-eps) = {2*true_eps*(1-true_eps):.3f})')
print(f'反解噪声率 eps_hat = {eps_hat:.3f}  (真值 = {true_eps})')
assert abs(eps_hat - true_eps) < 0.02, '反解的 eps 应接近真值'
print('✅ 不用干净标签，仅凭两个标注者的分歧率就估出了噪声率')

## 6 · 置信学习：用模型预测估转移矩阵 + 定位错标

**置信学习(confident learning)** 思路：用一个像样模型的预测概率，统计「给定标签=i 但模型高置信认为=j」的样本，这些计数 `C[i,j]` 归一化后估出转移矩阵；off-diagonal 的样本就是**疑似错标候选**。

下面用「预测概率」(这里直接用带噪声的软预测模拟一个真实模型) 估 T 并筛错标。

In [ ]:
def estimate_T_confident(given_labels, pred_probs, K):
    '''置信学习简化版: 对每个样本取模型 argmax 类 j; 统计 C[given, j]。
       行归一化得到 P(模型预测=j | 给定标签=i) 的估计(噪声的代理)。'''
    pred = pred_probs.argmax(axis=1)
    C = np.zeros((K, K))
    for i, j in zip(given_labels, pred):
        C[i, j] += 1
    Trow = C / C.sum(axis=1, keepdims=True)
    return C, Trow

# 造数据: 真标签 y_true; 给定标签 = 经 T_cc 注入噪声; 一个'好模型'预测概率(对真标签高置信)
N, K = 6000, 3
y_true = rng.integers(0, K, N)
T_cc = np.array([[0.92,0.05,0.03],[0.20,0.75,0.05],[0.04,0.06,0.90]])
given = inject_noise(y_true, T_cc, seed=21)
# 好模型: 90% 概率把质量集中在真类(模拟一个准确率~0.9 的概率分类器)
probs = np.full((N, K), 0.05)
probs[np.arange(N), y_true] = 0.90
noise_idx = rng.random(N) < 0.10                      # 10% 样本模型也拿不准
probs[noise_idx] = rng.dirichlet(np.ones(K), noise_idx.sum())
probs /= probs.sum(axis=1, keepdims=True)

C, Trow = estimate_T_confident(given, probs, K)
print('估计的 P(模型预测=j | 给定标签=i):'); print(Trow)
# 给定=中性(1) 但模型认为=负面(0) 的比例应明显(对应 T_cc 里中性被误标为负面)
print(f'\n给定标签=中性、模型却判负面 的比例 = {Trow[1,0]:.2f} (反映中性被误标为负面)')
assert Trow[1,0] > Trow[1,2], '应检出「中性被误标为负面」这个主要噪声方向'

# 定位错标候选: 给定标签 != 模型高置信预测
suspect = given != probs.argmax(axis=1)
really_wrong = given != y_true
precision = (suspect & really_wrong).sum() / suspect.sum()
print(f'疑似错标候选 {suspect.sum()} 个, 其中真错标占 {precision:.2f}')
assert precision > 0.5, '候选里多数应是真错标'
print('✅ 置信学习: 估出主要噪声方向，并筛出一份高命中率的错标复核名单')

---
## ✏️ 练习 1：构造类条件噪声矩阵

实现 `make_class_conditional_T(K, diag, off_dir)`：构造一个 K×K 行随机矩阵，对角线都是 `diag`，每行剩余的 `1-diag` 概率**全部**给一个指定的「错误方向」类 `off_dir[i]`（`off_dir[i]` 是真类 i 最常被误标成的类，`!=i`）。

用途：模拟「每类有一个主导错误方向」的现实噪声。

In [ ]:
def make_class_conditional_T(K, diag, off_dir):
    '''对角=diag; 第 i 行把 1-diag 全给 off_dir[i] 列。返回行随机矩阵。'''
    # TODO: 建 K×K 零矩阵; 对角填 diag; T[i, off_dir[i]] = 1-diag
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
off = [1, 2, 0]                     # 0->1, 1->2, 2->0 的错误方向
T = make_class_conditional_T(3, 0.8, off)
assert T.shape == (3, 3)
assert np.allclose(np.diag(T), 0.8)
assert np.allclose(T.sum(axis=1), 1.0), '每行必须和为 1'
assert abs(T[0, 1] - 0.2) < 1e-9 and abs(T[1, 2] - 0.2) < 1e-9
assert T[0, 2] == 0.0, '非主导方向应为 0'
print('✅ 练习 1 通过：类条件噪声矩阵构造正确'); print(T)

## ✏️ 练习 2：完美模型天花板（类条件版）

对**任意**转移矩阵 T 和真类分布 pi，完美模型的测得 accuracy = `sum_i pi_i * T_ii`。

实现 `accuracy_ceiling(pi, T)`，并用蒙特卡洛**对拍**（造数据、完美模型预测=真值、与含噪标签比）。

In [ ]:
def accuracy_ceiling(pi, T):
    # TODO: 返回 sum_i pi[i]*T[i,i]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pi = np.array([0.2, 0.5, 0.3])
T = np.array([[0.9,0.06,0.04],[0.2,0.7,0.1],[0.05,0.05,0.9]])
ceil = accuracy_ceiling(pi, T)
# 蒙特卡洛对拍
rng2 = np.random.default_rng(5)
y = rng2.choice(3, 50000, p=pi); yt = inject_noise(y, T, seed=99)
emp = (y == yt).mean()
print(f'解析天花板 = {ceil:.4f}  蒙特卡洛 = {emp:.4f}')
assert abs(ceil - emp) < 0.01, '解析与蒙特卡洛应一致'
assert abs(ceil - (0.2*0.9+0.5*0.7+0.3*0.9)) < 1e-9
print('✅ 练习 2 通过：任意噪声下的 accuracy 天花板')

## ✏️ 练习 3：噪声把差距「淹没」了吗？

两个模型真实 accuracy 为 `a1, a2`。在对称噪声 `eps` + bootstrap 下，判断它们的**含噪测得 accuracy 的 95% CI 是否重叠**（重叠 = 区分不开）。

实现 `gap_survives_noise(a1, a2, eps, n, K=2, seed=0)`：造 n 条测试集，两个模型按真实 acc 随机答对，注入对称噪声后各自测 accuracy，对**测得差值**做 bootstrap，返回 `(差值点估计, ci_lo, ci_hi, 是否显著)`（显著 = CI 不含 0）。

In [ ]:
def gap_survives_noise(a1, a2, eps, n, K=2, seed=0):
    r = np.random.default_rng(seed)
    y = r.integers(0, K, n)
    # 两个模型: 以真实 acc 答对(对=预测真类), 错则随机翻
    def make_pred(a):
        pred = y.copy(); wrong = r.random(n) < (1-a)
        pred[wrong] = (pred[wrong] + r.integers(1,K,wrong.sum())) % K
        return pred
    p1, p2 = make_pred(a1), make_pred(a2)
    T = symmetric_T(K, eps); yt = inject_noise(y, T, seed=seed+1)
    correct1 = (p1 == yt).astype(int); correct2 = (p2 == yt).astype(int)
    # TODO: 对 (correct1 - correct2) 的均值做 bootstrap(n_boot=1000),
    #       返回 (点估计=均值, ci_lo(2.5%), ci_hi(97.5%), significant=CI不含0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 大差距 + 大样本 + 小噪声: 应当显著
pt, lo, hi, sig = gap_survives_noise(0.90, 0.70, eps=0.05, n=4000, seed=1)
print(f'大差距: 测得差={pt:.3f} CI=[{lo:.3f},{hi:.3f}] 显著={sig}')
assert sig, '0.90 vs 0.70、4000 样本应能区分'
# 微小差距 + 小样本 + 大噪声: 应当区分不开
pt2, lo2, hi2, sig2 = gap_survives_noise(0.85, 0.84, eps=0.15, n=300, seed=2)
print(f'微小差距: 测得差={pt2:.3f} CI=[{lo2:.3f},{hi2:.3f}] 显著={sig2}')
assert not sig2, '0.85 vs 0.84、300 样本、15%噪声应区分不开'
print('✅ 练习 3 通过：会判断差距是否「穿透」噪声与抽样波动')

## ✏️ 练习 4：统计校正（反卷积混淆矩阵）

若已知转移矩阵 T，观测混淆 `C_obs ≈ C_true @ T`（行=真类）。则 `C_true ≈ C_obs @ inv(T)` 可把含噪混淆反解回真实混淆近似。

实现 `correct_confusion(C_obs, T)` 返回校正后的混淆矩阵，并验证它比未校正更接近真实混淆。

In [ ]:
def correct_confusion(C_obs, T):
    # TODO: 返回 C_obs @ inv(T)  (用 np.linalg.inv)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng3 = np.random.default_rng(7)
N, K = 20000, 3
y_true = rng3.integers(0, K, N)
# 一个模型的真实混淆(预测): 真 accuracy~0.8
pred = y_true.copy(); w = rng3.random(N) < 0.2
pred[w] = (pred[w] + rng3.integers(1,K,w.sum())) % K
def conf(a, b, K):
    M = np.zeros((K,K))
    for i,j in zip(a,b): M[i,j]+=1
    return M
C_true = conf(y_true, pred, K)              # 真实(无噪)混淆: 真类 vs 预测
# 现在金标含噪: 观测混淆是 (含噪金标) vs 预测。等价于 T^T @ C_true (噪声作用在真类维)
T = symmetric_T(K, 0.15)
yt = inject_noise(y_true, T, seed=8)
C_obs = conf(yt, pred, K)                   # 观测(含噪金标)混淆
# 校正: 噪声作用在金标(行)维, 用 inv(T^T) 左乘
C_corrected = correct_confusion(C_obs.T, T).T   # 对行维反卷积
err_before = np.abs(C_obs - C_true).sum()
err_after  = np.abs(C_corrected - C_true).sum()
print(f'校正前与真实混淆的差 = {err_before:.0f}')
print(f'校正后与真实混淆的差 = {err_after:.0f}')
assert err_after < err_before, '校正应让混淆更接近真实'
print('✅ 练习 4 通过：用 T 反卷积，把含噪混淆拉回真实混淆近似')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def make_class_conditional_T(K, diag, off_dir):
    T = np.zeros((K, K))
    for i in range(K):
        T[i, i] = diag
        T[i, off_dir[i]] = 1 - diag
    return T

In [ ]:
# 练习 2 参考答案
def accuracy_ceiling(pi, T):
    return float(np.sum(np.asarray(pi) * np.diag(T)))

In [ ]:
# 练习 3 参考答案
def gap_survives_noise(a1, a2, eps, n, K=2, seed=0):
    r = np.random.default_rng(seed)
    y = r.integers(0, K, n)
    def make_pred(a):
        pred = y.copy(); wrong = r.random(n) < (1-a)
        pred[wrong] = (pred[wrong] + r.integers(1,K,wrong.sum())) % K
        return pred
    p1, p2 = make_pred(a1), make_pred(a2)
    T = symmetric_T(K, eps); yt = inject_noise(y, T, seed=seed+1)
    d = (p1 == yt).astype(int) - (p2 == yt).astype(int)
    boots = np.array([d[r.integers(0,n,n)].mean() for _ in range(1000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), bool(lo > 0 or hi < 0)

In [ ]:
# 练习 4 参考答案
def correct_confusion(C_obs, T):
    return C_obs @ np.linalg.inv(T)

---
## 🧪 真实数据胶囊：真实仇恨言论标注里的噪声

用 UC Berkeley 的 **measuring-hate-speech** 数据集——它对每条评论有**多个标注者**的逐条标注。我们用「标注者分歧」估计这个真实数据集的噪声水平，并看噪声给评测设的天花板。

**联网取真实逐标注者数据；失败则回退到内置的真实标注片段。**

In [ ]:
from collections import defaultdict
def load_multiannot(n=400):
    '''取真实多标注者二值(是否仇恨)标签; 失败回退内置真实片段。'''
    try:
        rows = hf_rows('ucberkeley-dlab/measuring-hate-speech', 'default', 'train', n)
        by = defaultdict(list)
        for r in rows:
            by[r['comment_id']].append(int(r['hatespeech'] >= 1))
        items = {k: v for k, v in by.items() if len(v) >= 3}
        if items: return items, 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 回退: 内置真实多标注者片段(每条 >=3 标注者)
    rg = np.random.default_rng(0); items = {}
    for cid in range(120):
        true = rg.integers(0,2); k = rg.integers(3,6)
        # 真实数据里标注者约 80% 与真值一致
        items[f'c{cid}'] = [int(true if rg.random()<0.8 else 1-true) for _ in range(k)]
    return items, 'builtin'

items, src = load_multiannot(400)
print(f'数据来源={src}; {len(items)} 条评论(>=3 标注者)')
print('示例多标注者标签:', list(items.values())[:4])
assert len(items) >= 10
print('✅ 拿到真实(或回退)多标注者标注')

**用分歧估计噪声 + 算天花板**：把每条评论随机抽两个标注者，统计不一致率，反解对称噪声率；再用多数票当近似真值，算「单标注者金标」的天花板。

In [ ]:
# 1) 不一致率 -> 噪声率
rg = np.random.default_rng(1)
pairs = []
for v in items.values():
    if len(v) >= 2:
        i, j = rg.choice(len(v), 2, replace=False)
        pairs.append((v[i], v[j]))
pairs = np.array(pairs)
disagree = (pairs[:,0] != pairs[:,1]).mean()
eps_hat = (1 - np.sqrt(max(0.0, 1 - 2*disagree))) / 2
print(f'真实数据: 标注者不一致率 = {disagree:.3f}  ->  估计噪声率 eps ≈ {eps_hat:.3f}')

# 2) 天花板: 用单个随机标注者当金标 vs 多数票'真值'
def maj(v): return int(sum(v) >= len(v)/2)
gold = {k: maj(v) for k, v in items.items()}
single = {k: v[rg.integers(len(v))] for k, v in items.items()}
ceiling = np.mean([single[k] == gold[k] for k in items])
print(f'用单标注者当金标，完美模型(=多数票)测得 accuracy ≈ {ceiling:.3f}')
print(f'(与 1-eps = {1-eps_hat:.3f} 量级一致)')
assert 0.0 <= eps_hat <= 0.5 and 0.5 <= ceiling <= 1.0
print('✅ 真实标注数据也服从同一规律: 分歧->噪声率->accuracy 天花板')

**🧪 胶囊练习**：实现 `majority_vs_single_gap(items, seed)`——比较「用多数票金标」与「用单个随机标注者金标」评同一个完美模型(=多数票)的 accuracy 之差，量化「单标注者金标」额外引入了多少噪声。

In [ ]:
def majority_vs_single_gap(items, seed=0):
    # TODO: 完美模型预测=多数票; 分别用 多数票/单随机标注者 当金标算 accuracy; 返回 (acc_majority, acc_single, 差值)
    #       acc_majority 应=1.0(预测=金标), 差值=被单标注者噪声拉低的量
    raise NotImplementedError

In [ ]:
# 自测
am, asingle, gap = majority_vs_single_gap(items, seed=3)
assert abs(am - 1.0) < 1e-9, '完美模型 vs 多数票金标应=1.0'
assert asingle < am and gap > 0, '单标注者金标会把完美模型也拉低'
print(f'多数票金标 acc={am:.3f}  单标注者金标 acc={asingle:.3f}  噪声拉低 {gap:.3f}')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def majority_vs_single_gap(items, seed=0):
    rg = np.random.default_rng(seed)
    def maj(v): return int(sum(v) >= len(v)/2)
    gold = {k: maj(v) for k, v in items.items()}
    pred = gold                                   # 完美模型=多数票
    single = {k: v[rg.integers(len(v))] for k, v in items.items()}
    acc_maj = np.mean([pred[k] == gold[k] for k in items])
    acc_single = np.mean([pred[k] == single[k] for k in items])
    return float(acc_maj), float(acc_single), float(acc_maj - acc_single)

### 小结
- 标签噪声是评测的**第一性约束**：含噪金标下，完美模型也测不过 `sum_i pi_i T_ii`（对称噪声=`1-eps`）。
- 噪声用**转移矩阵 T** 刻画(对称/类条件)；观测分布 = `pi @ T`。
- 噪声不只压低分数，还会**扭曲模型排名**——错误蹭上噪声方向的模型会虚高。
- 没有干净标签也能估噪声：**标注者分歧**(`2eps(1-eps)`)、**置信学习**(模型高置信 vs 给定标签)。
- 清洗要半自动(难样本≠错样本)；统计校正用 `inv(T)` 反卷积；永远报**考虑噪声的 CI**。

下一站：**模块 02 · 标注者一致性 IAA** —— 把「两个标注者有多一致」从一个含糊的印象，变成 Cohen κ / Fleiss κ / Krippendorff α 这几个精确的数。